# browser-ai: SmolLM2 Personality Trainer

Fine-tune SmolLM2-360M-Instruct on any text to create a unique AI personality.
Uses Unsloth for fast LoRA training on Google Colab's free T4 GPU.

**Workflow:**
1. Upload your training text (TXT file)
2. Run training (~5-10 minutes on T4)
3. Download the merged model
4. Serve locally: `python train/serve.py --model ./output/personality`
5. Chat from your browser

In [ ]:
# Install dependencies
!pip install unsloth trl accelerate torch transformers datasets
!pip install optimum[onnx]  # for ONNX export

In [ ]:
# Clone the browser-ai repo
!git clone https://github.com/batraaryan03/browser-ai.git
%cd browser-ai

In [ ]:
# Upload your training text file
from google.colab import files
print("Upload your .txt file (50K+ characters recommended):")
uploaded = files.upload()

# Get the filename
import os
data_file = list(uploaded.keys())[0]
print(f"Using: {data_file}")

In [ ]:
# Run training
!python train/smol_lora_train.py \
  --data "{data_file}" \
  --epochs 3 \
  --steps 60 \
  --batch-size 2 \
  --lora-rank 16 \
  --output ./output

In [ ]:
# Download the merged model files
from google.colab import files
import zipfile
import os

model_dir = "./output/personality"
zip_path = "./output/personality-model.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(model_dir):
        for file in filenames:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, model_dir)
            zf.write(file_path, arcname)

print(f"Downloading {zip_path}...")
files.download(zip_path)
print("Done! Download the metadata.json too:")
files.download("./output/metadata.json")

## Next Steps

1. Unzip `personality-model.zip` on your computer
2. Install the serve dependencies: `pip install fastapi uvicorn transformers torch`
3. Run the server: `python train/serve.py --model ./output/personality`
4. Open your browser to the browser-ai Chat page

Your model is now running locally. Share your metadata.json with others
so they can experience the same personality!